# SQL Single-Table Query Strategies

**Purpose:** This guide is a quick-reference strategy sheet for SQL interview questions that involve transforming, analyzing, or reshaping data within a single table. It covers the most common patterns you'll encounter — comparing rows, aggregating, ranking, handling dates, pivoting, and more — with a focus on identifying the right approach before writing any code, choosing the most efficient function, and explaining your reasoning clearly to an interviewer.

<hr style="border: 3px solid black;">

## Table of Contents

1. [First Step: Identify the Pattern](#1-first-step-identify-the-pattern)
2. [Decision Tree (5-Second Version)](#2-decision-tree-5-second-version)
3. [Pattern Library with Examples](#3-pattern-library-with-examples)
    - [A. Row-Level Filtering](#pattern-a-row-level-filtering) — WHERE + ORDER BY
    - [B. Compare Rows](#pattern-b-compare-rows) — LAG / LEAD / Self Join
    - [C. Date Operations](#pattern-c-date-operations) — EXTRACT / DATEDIFF / INTERVAL
    - [D. Combine Rows](#pattern-d-combine-rows) — GROUP BY + CASE / Self Join
    - [E. Ranking](#pattern-e-ranking) — ROW_NUMBER / RANK / DENSE_RANK
    - [F. Running Totals](#pattern-f-running-totals) — SUM() OVER() / Correlated Subquery
    - [G. Missing Data](#pattern-g-missing-data) — NOT EXISTS / LEFT JOIN + IS NULL
    - [H. Deduplication](#pattern-h-deduplication) — ROW_NUMBER / DISTINCT ON / GROUP BY
    - [I. Conditional Logic](#pattern-i-conditional-logic) — CASE WHEN
    - [J. Filtering After Aggregation](#pattern-j-filtering-after-aggregation) — HAVING / Self Join / CTE + Join
    - [K. Pivoting](#pattern-k-pivoting) — CASE + GROUP BY
4. [Critical Efficiency Rules](#4-critical-efficiency-rules)
5. [GROUP BY vs Window Functions](#5-group-by-vs-window-functions)
6. [Method Selection: How to Know All Your Options](#6-method-selection-framework) — CTE vs Subquery, 3-Method Mental Model, Efficiency Rankings
7. [How to Think from the Vignette](#7-how-to-think-from-the-vignette)
8. [Interview Prompt Templates](#8-interview-prompt-templates)
9. [What to Avoid](#9-what-to-avoid)
10. [Final Takeaway](#10-final-takeaway)

<hr style="border: 3px solid black;">

<a id='1-first-step-identify-the-pattern'></a>

## 1. First Step: Identify the Pattern

Before writing **any** SQL, ask yourself:

> *What is this question actually asking me to do?*

The table below combines **pattern recognition**, **data type cues**, **function selection**, and **efficiency ranking** into a single reference. Functions are listed from most efficient (1) to least efficient (3).

| Pattern | Signal Words | Common Data Types | Rank | Function | Rationale |
|---|---|---|---|---|---|
| **Filter rows** | where, find, report, not equal to, odd/even, greater/less than | Any type | 1 | `WHERE` + operators (`=`, `!=`, `>`, `AND`, `OR`, `MOD`) | Direct row selection; single scan with index use |
| | | | 2 | `CASE` inside `WHERE` | Useful for complex multi-condition logic |
| **Compare rows** | previous, next, yesterday, consecutive, following | Date/timestamp, ordered values | 1 | `LAG()` (look back) / `LEAD()` (look ahead) | Single pass; designed for ordered row comparison |
| | | | 2 | Self `JOIN` | Works but adds extra join overhead |
| | | | 3 | Correlated subquery | Re-executes for every row; slowest |
| **Date operations** | days between, month, year, age, interval, gap | Date/timestamp | 1 | `DATEDIFF()` / `DATE_PART()` / `EXTRACT()` | Built-in date math; single pass |
| | | | 2 | `INTERVAL` arithmetic (e.g., `+ INTERVAL '1 day'`) | Flexible but syntax varies by dialect |
| | | | 3 | Manual cast/subtraction | Error-prone; avoid when date functions exist |
| **Combine rows** | avg, sum, duration, per X | Numeric, categorical (start/end), IDs | 1 | `GROUP BY` (+ `CASE`) | Direct aggregation; no duplicate rows produced |
| | | | 2 | Self `JOIN` | Clean pairing but more work than GROUP BY |
| | | | 3 | Window function | Creates duplicates that need removal |
| **Rank rows** | top, latest, first, nth | Ordered values, date/timestamp | 1 | `ROW_NUMBER()` | Direct ranking with partition control |
| | | | 2 | `RANK()` / `DENSE_RANK()` | Use when ties matter; slightly more complex |
| | | | 3 | Correlated subquery | Runs per row; poor performance on large tables |
| **Running calc** | cumulative, rolling, moving | Numeric, date/timestamp | 1 | `SUM() OVER()` | Efficient window frame; single pass |
| | | | 2 | Self `JOIN` | Joins all preceding rows; heavier |
| | | | 3 | Correlated subquery | Re-sums for every row; very slow |
| **Existence** | no, missing, without, never | IDs, any type | 1 | `NOT EXISTS` | Handles NULLs correctly; stops at first match |
| | | | 2 | `LEFT JOIN` + `IS NULL` | Valid anti-join; slightly more verbose |
| | | | 3 | `NOT IN` | Fails silently when NULLs present in subquery |
| **Deduplicate** | unique, one per, most recent | IDs, date/timestamp | 1 | `ROW_NUMBER()` | Full control over which row to keep |
| | | | 2 | `DISTINCT` | Simple but no control over row selection |
| | | | 3 | Self `JOIN` | Overly complex for deduplication |
| **Conditional logic** | label, categorize, bucket, if/then, flag | Any type | 1 | `CASE WHEN` (standalone) | Row-level labeling; no aggregation needed |
| | | | 2 | `CASE` inside `COUNT` / `SUM` | Count or sum by category in one query |
| | | | 3 | Multiple queries with `WHERE` | Separate query per category; redundant work |
| **Filter groups** | more than, at least, groups where, having | Numeric (aggregated) | 1 | `GROUP BY` + `HAVING` | Filters groups after aggregation; designed for this |
| | | | 2 | Subquery + `WHERE` | Works but adds nesting; less readable |
| **Pivot** | rows to columns, side by side, crosstab | Categorical + numeric | 1 | `CASE` + `GROUP BY` (manual pivot) | Portable; works in all dialects |
| | | | 2 | `PIVOT` keyword (SQL Server / Oracle) | Cleaner syntax but dialect-specific |
| | | | 3 | Application-level pivot | Moves work outside SQL; last resort |

<hr style="border: 3px solid black;">

<a id='2-decision-tree-5-second-version'></a>

## 2. Decision Tree (5-Second Version)

Use this quick mental checklist when you first read a problem:

```
┌───────────────────────────────────────┐
│        READ THE QUESTION              │
│        What am I being asked to do?   │
└──────────────────┬────────────────────┘
                   │
                   ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Just filtering rows by      ├───────►│  WHERE + ORDER BY            │
│  conditions on columns?      │        │                              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Does it involve dates?      ├───────►│  See DATE FORK below         │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I comparing rows?        ├───────►│  LAG() / LEAD()              │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I collapsing rows?       ├───────►│  GROUP BY                    │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Filtering after grouping?   ├───────►│  GROUP BY + HAVING           │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Am I ranking rows?          ├───────►│  ROW_NUMBER() / RANK         │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Running calculation?        ├───────►│  SUM() OVER()                │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Labeling or categorizing?   ├───────►│  CASE WHEN                   │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Turning rows into columns?  ├───────►│  CASE + GROUP BY (pivot)     │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Finding missing data?       ├───────►│  NOT EXISTS                  │
└──────────┬───────────────────┘        └──────────────────────────────┘
           │ NO
           ▼
┌──────────────────────────────┐  YES   ┌──────────────────────────────┐
│  Removing duplicates?        ├───────►│  ROW_NUMBER / DISTINCT       │
└──────────────────────────────┘        └──────────────────────────────┘
```

### DATE FORK — What kind of date work?

```
┌──────────────────────────────────────────────────────────────────────┐
│                    What kind of date work?                           │
└──────┬───────────────────────┬───────────────────────┬───────────────┘
       ▼                       ▼                       ▼
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  Comparing   │        │  Extracting  │        │  Calculating │
│  consecutive │        │  parts?      │        │  difference? │
│  rows?       │        │              │        │              │
│  (yesterday, │        │  (per month, │        │  (days       │
│   next day)  │        │   per year)  │        │   between,   │
│              │        │              │        │   gaps)      │
└──────┬───────┘        └──────┬───────┘        └──────┬───────┘
       ▼                       ▼                       ▼
┌──────────────┐        ┌──────────────┐        ┌──────────────┐
│  LAG / LEAD  │        │  EXTRACT /   │        │  DATEDIFF /  │
│  + date math │        │  DATE_PART   │        │  subtraction │
│              │        │  + GROUP BY  │        │  + LAG/LEAD  │
└──────────────┘        └──────────────┘        └──────────────┘
```

**Key insight:** Date problems almost always combine with another pattern. The date fork tells you *which* date tool you need, then you still flow into the main tree for the structural pattern (GROUP BY, LAG, ROW_NUMBER, etc.).

<hr style="border: 3px solid black;">

<a id='3-pattern-library-with-examples'></a>

## 3. Pattern Library with Examples

<a id='pattern-a-row-level-filtering'></a>

### A. Row-Level Filtering (WHERE + ORDER BY)

**Signals:** "find rows where," "report all X that satisfy," "not equal to," "odd-numbered," "greater than," "exclude," "not boring"

---

**Best approach — `WHERE` + operators**

**Method:** Apply conditions directly on existing columns using `WHERE`. Combine multiple conditions with `AND` / `OR`. Use `ORDER BY` when the output needs sorting. No aggregation, no window functions — just selecting the rows that match.

**In plain language:** "Look at each row, check if it meets the criteria, and return the ones that do."

```sql
-- Find movies with odd-numbered ID and description not 'boring', sorted by rating
SELECT *
FROM Cinema
WHERE id % 2 = 1
  AND description != 'boring'
ORDER BY rating DESC;
```

---

**Common WHERE Operators**

| Operator | Purpose | Example |
|---|---|---|
| `=`, `!=` (or `<>`) | Exact match / exclusion | `description != 'boring'` |
| `>`, `<`, `>=`, `<=` | Numeric / date comparison | `rating >= 8.0` |
| `%` (MOD) | Odd/even, divisibility | `id % 2 = 1` (odd) |
| `AND`, `OR` | Combine conditions | `WHERE a > 5 AND b = 'x'` |
| `IN (...)` | Match any in a list | `status IN ('active', 'pending')` |
| `NOT IN (...)` | Exclude a list | `category NOT IN ('spam', 'test')` |
| `BETWEEN` | Range (inclusive) | `rating BETWEEN 7.0 AND 9.0` |
| `LIKE` / `NOT LIKE` | Pattern matching | `name LIKE 'A%'` (starts with A) |
| `IS NULL` / `IS NOT NULL` | NULL checks | `email IS NOT NULL` |

---

**How to recognize this pattern:**

The question gives you conditions on **existing columns** and asks you to return matching rows — no aggregation, no comparison between rows, no grouping. The output shape is the **same columns as the input** (or a subset), just fewer rows.

> **Interview tip:** These problems look trivially easy, but interviewers use them to check if you write clean, correct SQL under pressure. Pay attention to edge cases like NULLs in `!=` comparisons (NULL != 'boring' evaluates to NULL, not TRUE) and whether `MOD` syntax varies by dialect (`%` in MySQL/PostgreSQL, `MOD()` function in Oracle).

<a id='pattern-b-compare-rows'></a>

### B. Compare Rows

**Signals:** previous, yesterday, next, consecutive, following

---

**Best approach — Window Function: `LAG()` (look backward)**

**Method:** `LAG()` is a window function that lets you access the previous row's value without a join. You define the order with `OVER (ORDER BY ...)` and it hands you the value from the row right before the current one. Use `LAG()` when the question asks about what came before — "previous day," "yesterday," "prior month."

**In plain language:** The inner query adds two new columns to every row — the previous day's date and the previous day's temperature. Then the outer query simply checks: "Is today exactly one day after yesterday, and is today's temperature higher?" If both are true, we keep that row's ID.

```sql
SELECT id
FROM (
    SELECT
        id,
        recordDate,
        temperature,
        LAG(recordDate) OVER (ORDER BY recordDate) AS prev_date,
        LAG(temperature) OVER (ORDER BY recordDate) AS prev_temp
    FROM Weather
) t
WHERE recordDate = prev_date + INTERVAL '1 day'
  AND temperature > prev_temp;
```

---

**Also common — Window Function: `LEAD()` (look forward)**

**Method:** `LEAD()` is the mirror image of `LAG()` — instead of looking at the previous row, it looks at the next row. Use `LEAD()` when the question asks about what comes after — "next purchase," "following day," "will the customer return."

**In plain language:** For every row, we peek ahead at the next row's value. This is useful when you need to ask "what happens next?" For example, finding users whose next login was more than 30 days later — that gap tells you they churned.

```sql
SELECT
    user_id,
    login_date,
    LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) AS next_login,
    LEAD(login_date) OVER (PARTITION BY user_id ORDER BY login_date) - login_date AS days_until_next
FROM Logins;
```

---

**When to use LAG vs LEAD**

| Question asks about... | Use | Example |
|---|---|---|
| What came **before** this row | `LAG()` | "Was yesterday's temperature lower?" |
| What comes **after** this row | `LEAD()` | "When is the user's next login?" |
| The **gap** between consecutive rows | Either works | "How many days between orders?" |

**In plain language:** Think of it like standing in a line. `LAG()` lets you turn around and look at the person behind you. `LEAD()` lets you look at the person in front of you. Both are window functions, both use `OVER (ORDER BY ...)`, and the syntax is identical — the only difference is the direction.

---

**Alternative — Self Join**

**Method:** Join the table to itself by matching each row to the row from the day before. No window function needed, but the database has to do extra work to pair up the rows.

**In plain language:** We take two copies of the same Weather table and line them up so that each day in the first copy is matched to the day before it in the second copy. Then we just check which days were warmer than the day before.

```sql
SELECT w1.id
FROM Weather w1
JOIN Weather w2
  ON w1.recordDate = w2.recordDate + INTERVAL '1 day'
WHERE w1.temperature > w2.temperature;
```

<hr style="border: 2px solid black;">

<a id='pattern-c-date-operations'></a>

### C. Date Operations

**Signals:** days between, month, year, age, interval, gap, extract

Date columns show up constantly in single-table interview problems. These functions are often **combined** with other patterns (LAG, GROUP BY, ROW_NUMBER) rather than used alone.

---

**Extracting parts of a date — `EXTRACT()` / `DATE_PART()` / `YEAR()`, `MONTH()`, `DAY()`**

**Method:** These functions pull a specific component (year, month, day, hour) out of a date or timestamp column. The syntax varies by dialect but the idea is the same everywhere.

**In plain language:** If you have a column with full dates like `2024-03-15` and the question asks "how many orders per month," you need to break that date into its month component first, then group by it. Think of it like opening a date and pulling out just the piece you need.

```sql
-- PostgreSQL / standard SQL
SELECT EXTRACT(MONTH FROM order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY EXTRACT(MONTH FROM order_date);

-- MySQL
SELECT MONTH(order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY MONTH(order_date);

-- SQL Server
SELECT DATEPART(MONTH, order_date) AS order_month, COUNT(*) AS total_orders
FROM Orders
GROUP BY DATEPART(MONTH, order_date);
```

---

**Calculating differences between dates — `DATEDIFF()` / subtraction / `AGE()`**

**Method:** These calculate the gap between two dates. Often paired with `LAG()` or `LEAD()` to find the time between consecutive rows in the same table. Syntax varies heavily by dialect.

**In plain language:** When you need to answer "how many days between event A and event B," you need date difference. If both events live in the same row (like a start and end column), you just subtract. If they live in different rows (like consecutive logins), you first use `LAG()` or `LEAD()` to bring them onto the same row, then subtract.

```sql
-- PostgreSQL: subtract dates directly (returns integer days)
SELECT order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS days_since_last
FROM Orders;

-- MySQL: use DATEDIFF
SELECT DATEDIFF(order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)) AS days_since_last
FROM Orders;

-- SQL Server: use DATEDIFF with unit
SELECT DATEDIFF(DAY, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date), order_date) AS days_since_last
FROM Orders;
```

---

**Date arithmetic — `INTERVAL` / `DATE_ADD()` / `DATEADD()`**

**Method:** Add or subtract a specific amount of time from a date. Used when the question says things like "within 7 days" or "one day after."

**In plain language:** This is for when you need to shift a date forward or backward by a fixed amount — like checking if a row's date is exactly one day after another, or finding all records from the last 30 days.

```sql
-- PostgreSQL
WHERE recordDate = prev_date + INTERVAL '1 day'

-- MySQL
WHERE recordDate = DATE_ADD(prev_date, INTERVAL 1 DAY)

-- SQL Server
WHERE recordDate = DATEADD(DAY, 1, prev_date)
```

---

**Common interview pattern: dates + LAG/LEAD together**

Most date questions in interviews aren't purely about date math — they combine dates with row comparison. The date functions handle the "how far apart" part, and LAG/LEAD handle the "bring two rows together" part.

| Interview question sounds like... | Approach |
|---|---|
| "Days between consecutive logins" | `LAG()` + date subtraction |
| "Orders per month" | `EXTRACT(MONTH ...)` + `GROUP BY` |
| "Users inactive for 30+ days" | `LEAD()` + `DATEDIFF()` |
| "Was the previous day exactly yesterday?" | `LAG()` + `INTERVAL '1 day'` |
| "First order each year" | `EXTRACT(YEAR ...)` + `ROW_NUMBER()` |

<hr style="border: 2px solid black;">

<a id='pattern-d-combine-rows'></a>

### D. Combine Rows

**Signals:** average, duration, per X, total per group

---

**Best approach — Aggregate Function: `GROUP BY` + Conditional Aggregation with `CASE`**

**Method:** `GROUP BY` collapses multiple rows into one row per group. Inside the aggregation, `CASE` expressions act like an if/then switch — they pick out specific values (like the "start" timestamp vs the "end" timestamp) so you can combine them in one pass.

**In plain language:** The inner query groups every activity by machine and process, then uses `CASE` to grab the end timestamp and the start timestamp separately, and subtracts them to get the process time. The outer query then takes all those process times for each machine and averages them. Two levels of grouping, but the logic reads top to bottom: first get each process duration, then average them per machine.

```sql
SELECT
    machine_id,
    ROUND(AVG(process_time)::numeric, 3)
FROM (
    SELECT
        machine_id,
        process_id,
        MAX(CASE WHEN activity_type = 'end' THEN timestamp END) -
        MAX(CASE WHEN activity_type = 'start' THEN timestamp END) AS process_time
    FROM Activity
    GROUP BY machine_id, process_id
) t
GROUP BY machine_id;
```

---

**Alternative — Self JOIN (pair start and end rows directly)**

**Method:** Join the table to itself, matching start rows to their corresponding end rows. Then aggregate the differences. This avoids the CASE expression by explicitly pairing the rows.

**In plain language:** Take two copies of the Activity table — one filtered to "start" events, one to "end" events — and match them on machine and process. Subtract to get duration, then average.

```sql
SELECT
    s.machine_id,
    ROUND(AVG(e.timestamp - s.timestamp)::numeric, 3) AS processing_time
FROM Activity s
JOIN Activity e
    ON s.machine_id = e.machine_id
    AND s.process_id = e.process_id
    AND s.activity_type = 'start'
    AND e.activity_type = 'end'
GROUP BY s.machine_id;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | GROUP BY + CASE (conditional aggregation) | Single scan, no join overhead |
| 2 | Self JOIN + GROUP BY | Clean and readable, but join has a cost |
| 3 | Correlated subquery | Runs per row — avoid |

<hr style="border: 2px solid black;">

<a id='pattern-e-ranking'></a>

### E. Ranking

**Signals:** top, latest, first, nth

---

**Best approach — Window Function: `ROW_NUMBER()`**

**Method:** `ROW_NUMBER()` is a window function that assigns a sequential number to each row within a partition. `PARTITION BY` splits the data into groups (like one group per customer), and `ORDER BY` controls which row gets number 1 within each group.

**In plain language:** The inner subquery is adding a row number to every row for a given customer_id, starting with the latest date as number 1, the second latest as number 2, and so on. So when we get to the outer query and filter for `rn = 1`, we are keeping only the most recent order for each customer and throwing away all the older ones.

```sql
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY order_date DESC
           ) AS rn
    FROM Orders
) t
WHERE rn = 1;
```

---

**When to use `RANK()` or `DENSE_RANK()` instead**

**Method:** `RANK()` and `DENSE_RANK()` are also window functions, but they handle ties differently. `RANK()` skips numbers after a tie (1, 1, 3), while `DENSE_RANK()` does not skip (1, 1, 2). Use these when the problem says "top N" and ties should be included.

**In plain language:** If two customers placed orders on the exact same date and you want to keep both of them (not randomly pick one), use `RANK()` instead of `ROW_NUMBER()`. The rest of the query stays the same — just swap the function name.

<hr style="border: 2px solid black;">

<a id='pattern-f-running-totals'></a>

### F. Running Totals

**Signals:** cumulative, rolling, moving, running sum, year-to-date

---

**Best approach — Window Function: `SUM() OVER()`**

**Method:** `SUM() OVER(ORDER BY ...)` is a window function that computes a running (cumulative) sum. Because there is no `PARTITION BY`, it treats the entire result set as one group. The `ORDER BY` inside `OVER()` tells SQL to add up all the sales from the beginning up to the current row.

**In plain language:** For every row, SQL looks at all the rows from the top of the table down to the current row, sums them up, and writes that total next to the current row. Each row gets a bigger total because it includes everything before it.

```sql
SELECT
    sale_date,
    amount,
    SUM(amount) OVER (ORDER BY sale_date) AS running_total
FROM Sales;
```

---

**Alternative — Correlated Subquery**

**Method:** For each row, run a subquery that sums all rows up to and including the current row. Produces the same result but runs the sum once per row.

**In plain language:** "For each sale, go back and add up every sale that happened on or before this date." On 1,000 rows, that's 1,000 separate sum calculations.

```sql
SELECT
    s1.sale_date,
    s1.amount,
    (SELECT SUM(s2.amount)
     FROM Sales s2
     WHERE s2.sale_date <= s1.sale_date) AS running_total
FROM Sales s1
ORDER BY s1.sale_date;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | `SUM() OVER()` (window function) | Single scan, built-in optimization |
| 2 | Correlated subquery | Runs sum per row — O(n²) on large tables |

<hr style="border: 2px solid black;">

<a id='pattern-g-missing-data'></a>

### G. Missing Data

**Signals:** no, missing, without, never, doesn't exist

---

**Best approach — Subquery Filter: `NOT EXISTS`**

**Method:** `NOT EXISTS` is a subquery filter (not a window function). It checks whether a matching row exists in another table. If no match is found, the row from the outer query is kept. It stops searching as soon as it finds the first match, making it efficient.

**In plain language:** We start with every customer in the Customers table. For each one, we peek into the Orders table and ask "does this customer have any orders?" If the answer is no (NOT EXISTS), we keep that customer. The subquery short-circuits — it stops as soon as it finds even one order.

```sql
SELECT customer_id, customer_name
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1
    FROM Orders o
    WHERE o.customer_id = c.customer_id
);
```

---

**Also strong — `LEFT JOIN` + `WHERE IS NULL`**

**Method:** Join the table to the reference table with a LEFT JOIN, then filter where the joined column is NULL. This identifies rows with no match.

**In plain language:** "Attach orders to each customer. Customers with no orders will have NULL in the order columns. Keep only those."

```sql
SELECT c.customer_id, c.customer_name
FROM Customers c
LEFT JOIN Orders o
    ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL;
```

---

**Avoid — `NOT IN` (with NULLs)**

**Method:** Compare against a list from a subquery. Dangerous if the subquery can return NULLs — the entire NOT IN silently returns zero rows.

**In plain language:** "Check if the ID is NOT IN the list of IDs from Orders." Sounds simple, but if any order has a NULL customer_id, the whole thing breaks and returns nothing.

```sql
-- DANGEROUS if customer_id in Orders can be NULL
SELECT customer_id, customer_name
FROM Customers
WHERE customer_id NOT IN (SELECT customer_id FROM Orders);
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | `NOT EXISTS` | Short-circuits on first match, NULL-safe |
| 2 | `LEFT JOIN` + `WHERE IS NULL` | Also efficient, NULL-safe, widely used |
| 3 | `NOT IN` | Breaks with NULLs — avoid unless column is guaranteed NOT NULL |

<hr style="border: 2px solid black;">

<a id='pattern-h-deduplication'></a>

### H. Deduplication

**Signals:** unique, one per, most recent per, latest per group, remove duplicates keeping one

---

**Best approach — Window Function: `ROW_NUMBER()`**

**Method:** Same window function as ranking — `ROW_NUMBER()` with `PARTITION BY` and `ORDER BY`. The difference is intent: here we are not ranking for display, we are numbering rows so we can throw away the duplicates and keep only the one we want (usually the most recent).

**In plain language:** The inner subquery looks at all login records for each user and numbers them starting from the most recent. The outer query keeps only row number 1 — the latest login per user.

```sql
SELECT user_id, login_date, device
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date DESC) AS rn
    FROM Logins
) t
WHERE rn = 1;
```

---

**Alternative — `GROUP BY` + aggregate (when you only need one column)**

**Method:** If you just need the max or min value per group (not the full row), GROUP BY with MAX/MIN is simpler and avoids the window function overhead.

**In plain language:** "Give me the most recent login date per user." No need to number rows — just take the MAX.

```sql
SELECT user_id, MAX(login_date) AS latest_login
FROM Logins
GROUP BY user_id;
```

---

**Alternative — `DISTINCT ON` (PostgreSQL only)**

**Method:** PostgreSQL's `DISTINCT ON` keeps the first row per group based on an ORDER BY. It's a shortcut for the ROW_NUMBER pattern.

**In plain language:** "For each user, keep only the first row when sorted by login date descending."

```sql
-- PostgreSQL only
SELECT DISTINCT ON (user_id) user_id, login_date, device
FROM Logins
ORDER BY user_id, login_date DESC;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | `DISTINCT ON` (PostgreSQL) | Optimized shortcut, single pass |
| 2 | `ROW_NUMBER()` + filter | Works everywhere, single scan |
| 3 | `GROUP BY` + `MAX/MIN` | Only works when you need the aggregate value, not the full row |
| 4 | Correlated subquery | Runs per row — avoid for dedup |

<hr style="border: 2px solid black;">

<a id='pattern-i-conditional-logic'></a>

### I. Conditional Logic (CASE Expressions)

**Signals:** label, categorize, bucket, if/then, classify, flag, convert rows to columns

---

**Standalone CASE — Categorizing or Bucketing Values**

**Method:** `CASE WHEN ... THEN ... ELSE ... END` is SQL's if/then/else. It evaluates conditions row by row and returns a value based on the first match. Use it standalone in SELECT to create new label columns, or inside WHERE/ORDER BY for conditional filtering and sorting.

**In plain language:** You're looking at each row and sticking a label on it. If the score is above 90, call it "high." If it's between 50 and 90, call it "medium." Everything else is "low." The table stays the same size — you're just adding a new column with your labels.

```sql
SELECT
    user_id,
    score,
    CASE
        WHEN score >= 90 THEN 'high'
        WHEN score >= 50 THEN 'medium'
        ELSE 'low'
    END AS score_tier
FROM Users;
```

---

**CASE inside COUNT / SUM — Counting by Category**

**Method:** Wrapping `CASE` inside an aggregate function like `COUNT()` or `SUM()` lets you count or sum only the rows that match a condition. This is a very common interview pattern for getting counts of different categories in a single query.

**In plain language:** Instead of running three separate queries to count how many users are "high," "medium," and "low," you do it all at once. For each row, `CASE` asks "does this match?" — if yes, it returns 1 (which gets counted), if no, it returns NULL (which gets skipped).

```sql
SELECT
    department,
    COUNT(CASE WHEN status = 'active' THEN 1 END) AS active_count,
    COUNT(CASE WHEN status = 'inactive' THEN 1 END) AS inactive_count,
    COUNT(*) AS total_count
FROM Employees
GROUP BY department;
```

<hr style="border: 2px solid black;">

<a id='pattern-j-filtering-after-aggregation'></a>

### J. Filtering After Aggregation (HAVING)

**Signals:** groups with more than, only customers who bought at least, categories where the average exceeds, managers with at least N reports

---

**Best approach — `GROUP BY` + `HAVING` (direct aggregation)**

**Method:** `HAVING` filters rows *after* `GROUP BY` has collapsed them. `WHERE` filters individual rows before grouping; `HAVING` filters the groups themselves based on aggregate values. You cannot use `WHERE` to filter on `COUNT()`, `SUM()`, `AVG()`, etc. — that's what `HAVING` is for.

**In plain language:** First, GROUP BY creates one row per group. Then HAVING looks at each group and asks "does this group meet my condition?" For example, "only keep customers who have more than 3 orders." WHERE can't do this because at the time WHERE runs, the rows haven't been grouped yet — it doesn't know what the count is.

```sql
-- Customers with more than 3 orders
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM Orders
GROUP BY customer_id
HAVING COUNT(*) > 3;
```

---

**Also strong — Self JOIN + GROUP BY + HAVING**

**Method:** When the output needs columns that aren't in the GROUP BY (like a name from a different row), join the table to itself first, then aggregate. This avoids a separate subquery step.

**In plain language:** Treat the table as two copies — one for the "parent" role and one for the "child" role. Join them, then count and filter. This reads like English: "find managers, count their reports, keep those with 5+."

```sql
-- Managers with at least 5 direct reports (single Employee table)
SELECT m.name
FROM Employee m
JOIN Employee r
    ON m.id = r.managerId
GROUP BY m.id, m.name
HAVING COUNT(*) >= 5;
```

---

**Also valid — Aggregate in CTE/Subquery, then JOIN back**

**Method:** First aggregate and filter in a CTE or subquery, then join the result back to the original table to pick up additional columns (like names). This separates the "find qualifying IDs" step from the "get their details" step.

**In plain language:** Step 1: count reports per manager and keep only those with 5+. Step 2: use those manager IDs to look up names. Two clean steps.

```sql
-- CTE version
WITH qualifying_managers AS (
    SELECT managerId AS id
    FROM Employee
    WHERE managerId IS NOT NULL
    GROUP BY managerId
    HAVING COUNT(*) >= 5
)
SELECT e.name
FROM qualifying_managers AS qm
JOIN Employee AS e
    ON qm.id = e.id;
```

---

**Avoid — Correlated Subquery**

**Method:** For each row, run a separate COUNT query to check if that row qualifies. This works but runs the inner query once per row in the outer table.

**In plain language:** "For every single employee, go count how many people report to them." On a table with 10,000 employees, that's 10,000 separate count operations.

```sql
-- Works but slower on large data
SELECT name
FROM Employee e
WHERE (
    SELECT COUNT(*)
    FROM Employee r
    WHERE r.managerId = e.id
) >= 5;
```

---

**Efficiency ranking for this pattern:**

| Rank | Method | Why |
|---|---|---|
| 1 | Self JOIN + GROUP BY + HAVING | Single query, no subquery, elegant |
| 2 | Aggregate in CTE/subquery + JOIN back | Equally efficient, cleaner separation of logic |
| 3 | Correlated subquery | Runs count per row — slower on large tables |

---

**WHERE vs HAVING — When to Use Which**

| Clause | Filters on | Runs when | Example |
|---|---|---|---|
| `WHERE` | Individual rows (before grouping) | Before `GROUP BY` | `WHERE status = 'active'` |
| `HAVING` | Aggregated groups (after grouping) | After `GROUP BY` | `HAVING COUNT(*) > 3` |

**In plain language:** Think of it as two gates. The first gate (WHERE) decides which individual rows are allowed into the grouping party. The second gate (HAVING) looks at the groups that formed and decides which groups are allowed into the final result. You can use both in the same query — WHERE narrows the rows first, then GROUP BY collapses them, then HAVING narrows the groups.

```sql
-- "Active customers who placed more than 5 orders in 2024"
SELECT
    customer_id,
    COUNT(*) AS order_count
FROM Orders
WHERE status = 'completed'                        -- gate 1: only completed orders
  AND EXTRACT(YEAR FROM order_date) = 2024        -- gate 1: only 2024
GROUP BY customer_id
HAVING COUNT(*) > 5;                              -- gate 2: only groups with 5+
```

<hr style="border: 2px solid black;">

<a id='pattern-k-pivoting'></a>

### K. Pivoting (Rows to Columns)

**Signals:** show each category as a column, side by side, pivot, crosstab, transpose

---

**Pivoting with CASE + GROUP BY — Turning Rows into Columns**

**Method:** Combine `CASE` expressions inside aggregate functions with `GROUP BY` to transform row values into separate columns. This is the most portable approach — it works in every SQL dialect. Some dialects also have a dedicated `PIVOT` keyword, but the CASE approach is what interviewers typically expect.

**In plain language:** Imagine you have a table where each row is a student's score for a different subject (one row for Math, one for Science, one for English). The interviewer wants you to show one row per student with Math, Science, and English as separate columns. You use CASE to say "if the subject is Math, grab the score" and wrap it in MAX or SUM so each CASE produces one value per student.

```sql
SELECT
    student_id,
    MAX(CASE WHEN subject = 'Math' THEN score END) AS math_score,
    MAX(CASE WHEN subject = 'Science' THEN score END) AS science_score,
    MAX(CASE WHEN subject = 'English' THEN score END) AS english_score
FROM Scores
GROUP BY student_id;
```

---

**Why MAX() around the CASE?**

**In plain language:** After GROUP BY collapses the rows, each group might have multiple rows (one per subject). The CASE picks the score only when the subject matches and returns NULL for the rest. MAX() grabs that one non-NULL value from the group. You could also use MIN() or SUM() — it doesn't matter when there's only one non-NULL value per group. The point is you need *some* aggregate function to satisfy GROUP BY.

---

**COUNT(DISTINCT ...) — Counting Unique Values in Groups**

**Signals:** how many different, number of unique, distinct count per group

**Method:** `COUNT(DISTINCT column)` inside a GROUP BY counts only the unique values of that column within each group. Regular `COUNT(*)` counts all rows; `COUNT(DISTINCT ...)` deduplicates before counting.

**In plain language:** If a customer bought the same product 3 times, `COUNT(*)` would say 3 orders but `COUNT(DISTINCT product_id)` would say 1 unique product. This comes up when the question asks "how many *different* products" rather than "how many orders."

```sql
SELECT
    customer_id,
    COUNT(*) AS total_orders,
    COUNT(DISTINCT product_id) AS unique_products
FROM Orders
GROUP BY customer_id;
```

<hr style="border: 3px solid black;">

<a id='4-critical-efficiency-rules'></a>

## 4. Critical Efficiency Rules

### Rule 1 — Avoid Duplicates

| Bad | Good |
|---|---|
| Window + `DISTINCT` | `GROUP BY` |

Using `DISTINCT` after a window function means the window computed values for rows that will be thrown away. Use `GROUP BY` to aggregate upfront.

### Rule 2 — Avoid Repeated Work

| Bad | Good |
|---|---|
| Correlated subquery | Window / Join |

Correlated subqueries re-execute for every row in the outer query. Window functions and joins compute results in a single pass.

### Rule 3 — Match Tool to Structure

| Need | Use |
|---|---|
| Previous row | `LAG` |
| One row per group | `GROUP BY` |
| Keep all rows | Window function |
| Pair rows | Join |

<hr style="border: 3px solid black;">

<a id='5-group-by-vs-window-functions'></a>

## 5. GROUP BY vs Window Functions

This is one of the **most important concepts** to understand for SQL interviews.

| Feature | GROUP BY | Window Function |
|---|---|---|
| **Effect on rows** | Collapses (reduces) rows | Keeps all rows |
| **Output** | 1 row per group | All original rows + new column |
| **Use when** | You need aggregated results only | You need row-level + group-level data |

### Example: GROUP BY Result

| machine | avg_time |
|---|---|
| A | 3.5 |
| B | 2.1 |

### Example: Window Function Result

| machine | process | time | avg_time |
|---|---|---|---|
| A | 1 | 3.0 | 3.5 |
| A | 2 | 4.0 | 3.5 |
| B | 1 | 2.1 | 2.1 |

### Memory Rule

| Rule | Meaning |
|---|---|
| Look across rows → **Window** | Comparisons |
| Collapse rows → **GROUP BY** | Aggregation |
| Pair rows → **Join** | Start/end matching |
| Keep rows + add info → **Window** | Group context |
| Need anti-match → **NOT EXISTS** | Missing data |

<hr style="border: 3px solid black;">

<a id='6-method-selection-framework'></a>

## 6. Method Selection: How to Know All Your Options

Once you identify the pattern, there are almost always **multiple valid SQL approaches**. The key skill isn't just picking the right pattern — it's knowing all the methods available for that pattern and choosing the most efficient one.

### The 3-Method Mental Model

For nearly every single-table pattern, your options fall into three categories:

| # | Method Type | How It Works | Typical Efficiency |
|---|---|---|---|
| 1 | **Direct aggregation** (`GROUP BY` + aggregate functions) | Collapse rows in one pass | Best — single scan |
| 2 | **Window function** (`OVER()`) | Compute across rows without collapsing | Great — single scan, keeps all rows |
| 3 | **Correlated subquery** | Re-run a subquery for every row | Weakest — runs N times for N rows |

### How to Enumerate Your Options — Decision Flow

Use this flow after you've identified the pattern. It tells you which methods are available and which to pick:

```
┌─────────────────────────────────────────┐
│  I identified the pattern.              │
│  Now: what are my method options?       │
└────────────────────┬────────────────────┘
                     │
                     ▼
┌──────────────────────────────────────┐
│  Does the output have FEWER rows     │
│  than the input? (collapsing)        │
└──────────┬───────────────┬───────────┘
           │ YES           │ NO
           ▼               ▼
┌────────────────────┐  ┌──────────────────────────────┐
│  Start with        │  │  Use a WINDOW FUNCTION       │
│  GROUP BY          │  │  (keeps all rows,            │
│  (most efficient)  │  │   adds computed column)      │
└─────────┬──────────┘  └──────────────────────────────┘
          │
          ▼
┌──────────────────────────────────────┐
│  Does the output need columns that   │
│  aren't in the GROUP BY?             │
│  (e.g., names, details)              │
└──────────┬───────────────┬───────────┘
           │ YES           │ NO
           ▼               ▼
┌────────────────────┐  ┌──────────────────────────────┐
│  Need to JOIN back │  │  Done! GROUP BY + HAVING     │
│  to get extra cols │  │  is your complete answer.    │
└─────────┬──────────┘  │  Rank 1: Most efficient.     │
          │             └──────────────────────────────┘
          ▼
┌──────────────────────────────────────┐
│  Pick your JOIN-back strategy:       │
├──────────────────────────────────────┤
│                                      │
│  Option A (Rank 1):                  │
│  Self JOIN + GROUP BY + HAVING       │
│  → Join first, then aggregate        │
│  → One query, reads like English     │
│                                      │
│  Option B (Rank 2):                  │
│  CTE/Subquery + JOIN back            │
│  → Aggregate first, then join        │
│  → Clean separation of steps         │
│  → Same performance as Option A      │
│                                      │
│  Option C (Rank 3 — avoid):          │
│  Correlated subquery                 │
│  → Runs aggregate per row            │
│  → Mention it, then reject it        │
│                                      │
└──────────────────────────────────────┘
```

### CTE vs Subquery — They Are Wrappers, Not Strategies

A `WITH ... AS` (CTE) and an inline subquery do the **exact same work**. The CTE just gives it a name. PostgreSQL typically inlines CTEs, so performance is identical.

```sql
-- These two are equivalent in performance:

-- CTE version
WITH counts AS (
    SELECT department, COUNT(*) AS cnt
    FROM Employees
    GROUP BY department
)
SELECT * FROM counts WHERE cnt > 5;

-- Subquery version
SELECT * FROM (
    SELECT department, COUNT(*) AS cnt
    FROM Employees
    GROUP BY department
) counts WHERE cnt > 5;
```

**When to use a CTE:** When you reference the result more than once, or when it makes the query more readable. **It is not a separate strategy** — it is a container for any strategy.

### Pattern → All Methods → Pick Best (Worked Example)

**Problem:** "Find managers with at least 5 direct reports" (single Employee table with id, name, managerId)

**Step 1: Identify the pattern.** Filtering after aggregation → HAVING

**Step 2: Walk the flow diagram.**
- Output has fewer rows than input? **Yes** → Start with GROUP BY
- Output needs columns not in GROUP BY (manager name)? **Yes** → Need to join back
- Pick join-back strategy → evaluate all three options

**Step 3: Enumerate all methods:**

| Rank | Method | SQL Sketch | Why This Rank |
|---|---|---|---|
| 1 | Self JOIN + GROUP BY + HAVING | `JOIN Employee r ON m.id = r.managerId GROUP BY ... HAVING COUNT(*) >= 5` | One query, no subquery, reads like English |
| 2 | Aggregate in subquery/CTE, then JOIN back | `WITH mgrs AS (SELECT managerId ... HAVING COUNT(*) >= 5) SELECT name FROM mgrs JOIN Employee ...` | Clean separation of logic, equally efficient |
| 3 | Correlated subquery in WHERE | `WHERE (SELECT COUNT(*) FROM Employee r WHERE r.managerId = e.id) >= 5` | Runs the count per row — less scalable |

**Step 4: Choose.** Methods 1 and 2 are essentially equal in performance. Method 1 is more elegant for interviews. Method 3 works but is demonstrably slower on large data.

### General Efficiency Ranking

This ranking applies to **most** single-table patterns:

| Rank | Approach | When It Wins |
|---|---|---|
| 1 | `GROUP BY` (direct aggregation) | When you need collapsed results |
| 2 | Window function | When you need row-level + group-level data |
| 3 | Self JOIN + aggregate | When output needs columns from different "roles" of the same row |
| 4 | CTE/Subquery + JOIN back | When you aggregate first, then enrich with other columns |
| 5 | Correlated subquery | Almost never the best; useful as a fallback or for simple existence checks |

### Interview Tip

When explaining your approach, briefly mention the alternatives you considered:

> *"I'm using a self-join with GROUP BY and HAVING here. I could also pre-aggregate manager IDs in a CTE and join back for the name, which would be equally efficient. A correlated subquery would also work but would be slower on large data since it runs the count once per row."*

This shows the interviewer you understand the trade-offs, not just one way to get the answer.

<hr style="border: 3px solid black;">

<a id='7-how-to-think-from-the-vignette'></a>

## 7. How to Think from the Vignette

Follow these steps **before writing any SQL**:

### Step 1 — Identify the Output Shape

> *What should the final table look like?*

Examples:
- Weather problem → list of IDs (one per qualifying day)
- Activity problem → 1 row per machine

### Step 2 — Ask: Row vs Group?

> *Am I comparing rows or combining rows?*

- Comparing → Window function territory
- Combining → GROUP BY territory

### Step 3 — Identify the Grouping Level

> *What is the unit of output?*

Examples:
- Per machine → `GROUP BY machine_id`
- Per customer → `GROUP BY customer_id`
- No grouping → no `GROUP BY`

### Step 4 — Identify Pairing Logic

> *Do I need to match rows together?*

Examples:
- Start + end → `CASE` aggregation or self join
- Today + yesterday → `LAG`

### Step 5 — Choose the Tool

Use the [Decision Tree](#2-decision-tree-5-second-version) and the [Master Pattern Table](#1-first-step-identify-the-pattern) to select the best approach.

---

### Worked Examples

**Weather Problem:**

| Step | Answer |
|---|---|
| Output shape | List of IDs |
| Row vs group | Comparing rows |
| Grouping level | None |
| Pairing logic | Today vs yesterday |
| **Tool** | **LAG** |

**Activity Problem:**

| Step | Answer |
|---|---|
| Output shape | 1 row per machine |
| Row vs group | Combining rows |
| Grouping level | machine_id, process_id |
| Pairing logic | Start + end timestamps |
| **Tool** | **GROUP BY + CASE** |

<hr style="border: 3px solid black;">

<a id='8-interview-prompt-templates'></a>

## 8. Interview Prompt Templates

Use these phrases during an interview to structure your thinking out loud:

| # | Template | When to Use |
|---|---|---|
| 1 | *"This is a [compare/combine/rank] problem because..."* | Pattern identification |
| 2 | *"The output is one row per [X], so I'll group by [X]."* | Output shape |
| 3 | *"I need to compare each row to the previous one, so I'll use LAG."* | Row comparison |
| 4 | *"I need to collapse rows into one, so I'll use GROUP BY."* | Aggregation |
| 5 | *"I need to match related rows, so I'll join or use CASE aggregation."* | Pairing rows |
| 6 | *"This avoids duplicate computation and reduces rows early."* | Efficiency reasoning |
| 7 | *"Window functions let me keep all rows while adding group-level values."* | Window explanation |
| 8 | *"I won't use DISTINCT because it removes duplicates after computation."* | Avoiding mistakes |
| 9 | *"Does this produce exactly one row per required output?"* | Final check |
| 10 | *"If window functions weren't available, I'd use a self join."* | Backup strategy |

<hr style="border: 3px solid black;">

<a id='9-what-to-avoid'></a>

## 9. What to Avoid

| Bad Pattern | Why It's Bad | Better Alternative |
|---|---|---|
| Window + `DISTINCT` | Duplicate work — computes then discards | `GROUP BY` |
| Correlated subquery | Repeated work — runs per row | `LAG` / `JOIN` |
| `NOT IN` (with NULLs) | Incorrect results — NULLs break logic | `NOT EXISTS` |
| Self join for running totals | Heavy computation | `SUM() OVER()` |

<hr style="border: 3px solid black;">

<a id='10-final-takeaway'></a>

## 10. Final Takeaway

The entire game is:

```
Identify the shape  →  Pick the tool  →  Avoid unnecessary work
```

### Interview Script (Memorize This)

> *"I identify whether this is comparing rows or aggregating them. If it's row comparison, I use LAG. If it's aggregation, I use GROUP BY. If I need both row-level and group-level data, I use a window function."*